XGBoost installed ✅
Regular Season shape : (122775, 34)
Seeds shape          : (2626, 3)
Tourney Results shape: (1449, 34)
Regular seasons      : [np.int64(2003), np.int64(2004), np.int64(2005), np.int64(2006), np.int64(2007), np.int64(2008), np.int64(2009), np.int64(2010), np.int64(2011), np.int64(2012), np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025), np.int64(2026)]
Tourney seasons      : [np.int64(2003), np.int64(2004), np.int64(2005), np.int64(2006), np.int64(2007), np.int64(2008), np.int64(2009), np.int64(2010), np.int64(2011), np.int64(2012), np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]

Team features shape: (8346, 18)

Building training matchups...
  Regular season games (2005-

In [1]:

"""
=============================================================================
NCAA March Mania 2026 — FINAL SUBMISSION GENERATOR (FULLY FIXED)
=============================================================================
Produces Updated_submission_stage2_final_one.csv with exactly 132,133 rows.
Men   (TeamID 1xxx) : trained on M files, men's best BO params
Women (TeamID 3xxx) : trained on W files, women's best BO params

FIXES APPLIED:
  ① 2026 seeds used for prediction (falls back to 2025 if not available)
  ② Conference strength penalty — weak conferences get ELO penalty
  ③ Seed overrides — 1 vs 16 can never be < 97.5% for 1-seed
  ④ ID dict mapping — no reindex drift
=============================================================================
"""

import warnings; warnings.filterwarnings("ignore")
import os, multiprocessing, re
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.metrics import log_loss, accuracy_score, roc_auc_score
import lightgbm as lgb
import xgboost as xgb

# ─────────────────────────────────────────────────────────────
# CONFIG
# ─────────────────────────────────────────────────────────────
DATA_DIR = Path("/Users/shaurya/Downloads")
OUTPUT   = DATA_DIR / "shaurya_final_1.csv"  # FIX ④ unique name

N_JOBS = multiprocessing.cpu_count()
os.environ["OMP_NUM_THREADS"]      = str(N_JOBS)
os.environ["MKL_NUM_THREADS"]      = str(N_JOBS)
os.environ["OPENBLAS_NUM_THREADS"] = str(N_JOBS)
print(f"🖥️  Using {N_JOBS} CPU cores")

TRAIN_SEASONS = list(range(2010, 2026))   # 2010–2025 inclusive
ELO_K         = 20
ELO_BASE      = 1500
ELO_HOME_ADV  = 100
SEED          = 42
np.random.seed(SEED)

STAT_COLS = ["FGM","FGA","FGM3","FGA3","FTM","FTA",
             "OR","DR","Ast","TO","Stl","Blk","PF"]

# ─────────────────────────────────────────────────────────────
# FIX ③ — SEED OVERRIDES
# Forces extreme seed matchups to realistic probabilities.
# ─────────────────────────────────────────────────────────────
SEED_OVERRIDES_MENS = {
    (1, 16): 0.975,
    (1, 15): 0.945,
    (1, 14): 0.920,
    (1, 13): 0.900,
    (2, 15): 0.930,
    (2, 16): 0.960,
    (3, 14): 0.850,
    (4, 13): 0.820,
    (5, 12): 0.650,
    (6, 11): 0.620,
    (7, 10): 0.600,
    (8,  9): 0.520,
}
SEED_OVERRIDES_WOMENS = {
    (1, 16): 0.985,
    (1, 15): 0.965,
    (1, 14): 0.945,
    (1, 13): 0.925,
    (2, 15): 0.955,
    (2, 16): 0.975,
    (3, 14): 0.875,
    (4, 13): 0.850,
    (5, 12): 0.720,
    (6, 11): 0.680,
    (7, 10): 0.650,
    (8,  9): 0.540,
}

# ─────────────────────────────────────────────────────────────
# FIX ② — CONFERENCE STRENGTH TIERS
# Weak-conference teams have inflated stats vs weak opponents.
# ─────────────────────────────────────────────────────────────
POWER_CONFS = {
    "acc","big_east","big_ten","big_twelve","pac_twelve",
    "sec","aac","mwc","wcc","a_ten"
}
WEAK_CONFS = {
    "meac","swac","nec","big_south","independent"
}
CONF_ELO_PENALTY = {
    "power"    :    0,
    "mid_major":  -50,
    "weak"     : -150,
}

# ─────────────────────────────────────────────────────────────
# BEST PARAMS — MEN'S
# ─────────────────────────────────────────────────────────────
MEN_LGBM = {
    "num_leaves"        : 170,
    "max_depth"         : 12,
    "learning_rate"     : 0.005,
    "n_estimators"      : 1308 + 200,
    "min_child_samples" : 5,
    "subsample"         : 0.5,
    "colsample_bytree"  : 0.5,
    "reg_alpha"         : 0.0,
    "reg_lambda"        : 0.0,
    "objective"         : "binary",
    "metric"            : "binary_logloss",
    "random_state"      : SEED,
    "n_jobs"            : N_JOBS,
    "device_type"       : "cpu",
    "num_threads"       : N_JOBS,
    "verbose"           : -1,
}
MEN_XGB = {
    "n_estimators"          : 1638 + 200,
    "max_depth"             : 10,
    "learning_rate"         : 0.0255778,
    "min_child_weight"      : 2.84511,
    "subsample"             : 0.848826,
    "colsample_bytree"      : 0.562814,
    "gamma"                 : 1.59748,
    "reg_alpha"             : 4.86392,
    "reg_lambda"            : 1.04622,
    "early_stopping_rounds" : 50,
    "objective"             : "binary:logistic",
    "eval_metric"           : "logloss",
    "random_state"          : SEED,
    "n_jobs"                : N_JOBS,
    "verbosity"             : 0,
    "tree_method"           : "hist",
}

# ─────────────────────────────────────────────────────────────
# BEST PARAMS — WOMEN'S
# ─────────────────────────────────────────────────────────────
WOMEN_LGBM = {
    "num_leaves"        : 164,
    "max_depth"         : 8,
    "learning_rate"     : 0.0187029,
    "n_estimators"      : 1294 + 200,
    "min_child_samples" : 22,
    "subsample"         : 0.532526,
    "colsample_bytree"  : 0.974443,
    "reg_alpha"         : 4.82816,
    "reg_lambda"        : 4.04199,
    "objective"         : "binary",
    "metric"            : "binary_logloss",
    "random_state"      : SEED,
    "n_jobs"            : N_JOBS,
    "device_type"       : "cpu",
    "num_threads"       : N_JOBS,
    "verbose"           : -1,
}
WOMEN_XGB = {
    "n_estimators"          : 1898 + 200,
    "max_depth"             : 7,
    "learning_rate"         : 0.0559337,
    "min_child_weight"      : 18.9736,
    "subsample"             : 0.842968,
    "colsample_bytree"      : 0.895046,
    "gamma"                 : 0.171643,
    "reg_alpha"             : 1.13418,
    "reg_lambda"            : 2.44057,
    "early_stopping_rounds" : 50,
    "objective"             : "binary:logistic",
    "eval_metric"           : "logloss",
    "random_state"          : SEED,
    "n_jobs"                : N_JOBS,
    "verbosity"             : 0,
    "tree_method"           : "hist",
}

# ─────────────────────────────────────────────────────────────
# SHARED HELPER FUNCTIONS
# ─────────────────────────────────────────────────────────────
def parse_seed_num(s):
    if pd.isna(s): return 16
    m = re.search(r"\d+", str(s))
    return int(m.group()) if m else 16

def compute_elo(reg_df, tourney_df):
    cols = ["Season","DayNum","WTeamID","LTeamID","WLoc","WScore","LScore"]
    all_games = pd.concat([reg_df[cols], tourney_df[cols]]) \
                  .sort_values(["Season","DayNum"]).reset_index(drop=True)
    elo = {}
    season_end = {}
    for season in sorted(all_games["Season"].unique()):
        for tid in list(elo):
            elo[tid] = ELO_BASE * 0.25 + elo[tid] * 0.75
        for _, r in all_games[all_games["Season"] == season].iterrows():
            w, l   = int(r["WTeamID"]), int(r["LTeamID"])
            rw, rl = elo.get(w, ELO_BASE), elo.get(l, ELO_BASE)
            loc    = r["WLoc"]
            rw_adj = rw + ELO_HOME_ADV if loc=="H" \
                     else (rw - ELO_HOME_ADV if loc=="A" else rw)
            mov    = min(abs(int(r["WScore"]) - int(r["LScore"])), 20)
            mov_m  = np.log1p(mov) * (2.2 / ((rw_adj - rl) * 0.001 + 2.2))
            exp_w  = 1 / (1 + 10 ** ((rl - rw_adj) / 400))
            delta  = ELO_K * mov_m * (1 - exp_w)
            elo[w] = rw + delta
            elo[l] = rl - delta
        for tid, rat in elo.items():
            season_end[(season, tid)] = rat
    return season_end, elo

def explode_games(games):
    rw = {"WTeamID":"TeamID","LTeamID":"OppID",
          "WScore":"PtsFor","LScore":"PtsAgainst",
          **{f"W{s}":s for s in STAT_COLS},
          **{f"L{s}":f"Opp{s}" for s in STAT_COLS}}
    rl = {"LTeamID":"TeamID","WTeamID":"OppID",
          "LScore":"PtsFor","WScore":"PtsAgainst",
          **{f"L{s}":s for s in STAT_COLS},
          **{f"W{s}":f"Opp{s}" for s in STAT_COLS}}
    keep = ["Season","DayNum","TeamID","OppID","PtsFor","PtsAgainst"] + \
           STAT_COLS + [f"Opp{s}" for s in STAT_COLS]
    w = games.rename(columns=rw); w["Win"] = 1
    l = games.rename(columns=rl); l["Win"] = 0
    return pd.concat([w[keep+["Win"]], l[keep+["Win"]]], ignore_index=True)

def build_team_stats(reg_df, tourney_df):
    tg  = explode_games(pd.concat([reg_df, tourney_df]))
    eps = 1e-6
    agg = tg.groupby(["Season","TeamID"]).agg(
        GP=("Win","count"), Wins=("Win","sum"),
        PtsFor=("PtsFor","mean"), PtsAgainst=("PtsAgainst","mean"),
        FGM=("FGM","mean"), FGA=("FGA","mean"),
        FGM3=("FGM3","mean"), FGA3=("FGA3","mean"),
        FTM=("FTM","mean"), FTA=("FTA","mean"),
        OR=("OR","mean"),   DR=("DR","mean"),
        Ast=("Ast","mean"), TO=("TO","mean"),
        Stl=("Stl","mean"), Blk=("Blk","mean"), PF=("PF","mean"),
        OppFGM=("OppFGM","mean"),   OppFGA=("OppFGA","mean"),
        OppFGM3=("OppFGM3","mean"), OppFGA3=("OppFGA3","mean"),
        OppOR=("OppOR","mean"),     OppDR=("OppDR","mean"),
    ).reset_index()
    agg["WinPct"]         = agg["Wins"]    / (agg["GP"]      + eps)
    agg["FGPct"]          = agg["FGM"]     / (agg["FGA"]     + eps)
    agg["FG3Pct"]         = agg["FGM3"]    / (agg["FGA3"]    + eps)
    agg["FTPct"]          = agg["FTM"]     / (agg["FTA"]     + eps)
    agg["OppFGPct"]       = agg["OppFGM"]  / (agg["OppFGA"]  + eps)
    agg["eFGPct"]         = (agg["FGM"]    + 0.5*agg["FGM3"])    / (agg["FGA"]    + eps)
    agg["OppEFGPct"]      = (agg["OppFGM"] + 0.5*agg["OppFGM3"]) / (agg["OppFGA"] + eps)
    poss                  = agg["FGA"] - agg["OR"] + agg["TO"] + 0.44*agg["FTA"]
    agg["TOVRate"]        = agg["TO"]  / (poss              + eps)
    agg["ORBRate"]        = agg["OR"]  / (agg["OR"] + agg["OppDR"] + eps)
    agg["FTRate"]         = agg["FTM"] / (agg["FGA"]        + eps)
    agg["DRBRate"]        = agg["DR"]  / (agg["DR"] + agg["OppOR"] + eps)
    agg["Pace"]           = poss       / (agg["GP"]          + eps)
    agg["NetRtg"]         = agg["PtsFor"] - agg["PtsAgainst"]
    agg["OffRtg"]         = agg["PtsFor"]     / (poss + eps) * 100
    agg["DefRtg"]         = agg["PtsAgainst"] / (poss + eps) * 100
    agg["NetRtg2"]        = agg["OffRtg"] - agg["DefRtg"]
    agg["AstTO"]          = agg["Ast"] / (agg["TO"]  + eps)
    agg["StlBlk"]         = (agg["Stl"] + agg["Blk"]) / (agg["OppFGA"] + eps)
    agg["ThreePtRate"]    = agg["FGA3"]    / (agg["FGA"]    + eps)
    agg["OppThreePtRate"] = agg["OppFGA3"] / (agg["OppFGA"] + eps)
    return agg

def build_recent_form(reg_df, n=10):
    tg  = explode_games(reg_df).sort_values(["Season","TeamID","DayNum"])
    rec = tg.groupby(["Season","TeamID"]).tail(n)
    rf  = rec.groupby(["Season","TeamID"]).agg(
        RecentWinPct=("Win","mean"),
        RecentPtsFor=("PtsFor","mean"),
        RecentPtsAgainst=("PtsAgainst","mean"),
    ).reset_index()
    rf["RecentNetPts"] = rf["RecentPtsFor"] - rf["RecentPtsAgainst"]
    return rf

def build_massey(seasons):
    fp = DATA_DIR / "MMasseyOrdinals.csv"
    if not fp.exists(): return None
    df = pd.read_csv(fp)
    df = df[(df["Season"].isin(seasons)) & (df["RankingDayNum"] <= 128)]
    result = (df.groupby(["Season","TeamID"])["OrdinalRank"].mean()
                .reset_index().rename(columns={"OrdinalRank":"AvgRank"}))
    for sys in ["SAG","POM","WOL","MOR","DOK","REW","NET"]:
        sub = df[df["SystemName"] == sys]
        if len(sub) == 0: continue
        s = (sub.groupby(["Season","TeamID"])["OrdinalRank"].mean()
                .reset_index().rename(columns={"OrdinalRank":f"Rank_{sys}"}))
        result = result.merge(s, on=["Season","TeamID"], how="left")
    return result

# FIX ② — build conference penalty dict
def build_conf_penalty(conf_csv):
    fp = DATA_DIR / conf_csv
    if not fp.exists():
        print(f"  ⚠️  {conf_csv} not found — skipping conference penalty")
        return {}
    df = pd.read_csv(fp)
    df = df[df["Season"].isin(TRAIN_SEASONS)]
    penalty = {}
    for _, r in df.iterrows():
        conf = str(r["ConfAbbrev"]).lower()
        if conf in POWER_CONFS:
            p = CONF_ELO_PENALTY["power"]
        elif conf in WEAK_CONFS:
            p = CONF_ELO_PENALTY["weak"]
        else:
            p = CONF_ELO_PENALTY["mid_major"]
        penalty[(int(r["Season"]), int(r["TeamID"]))] = p
    return penalty

def apply_conf_penalty(elo_final, conf_penalty, season=2025):
    adjusted = {}
    for tid, rating in elo_final.items():
        pen = conf_penalty.get((season, tid), CONF_ELO_PENALTY["mid_major"])
        adjusted[tid] = rating + pen
    return adjusted

# FIX ③ — apply seed overrides
def apply_seed_overrides(preds, ids, seed_map, seed_overrides):
    adj = preds.copy()
    print(f"          Seed map has {len(seed_map)} teams")
    n_missing = 0
    for idx, row_id in enumerate(ids):
        parts = row_id.split("_")
        t1, t2 = int(parts[1]), int(parts[2])
        # Default missing teams to 16 (not 8) so 1-seeds always trigger override
        s1 = seed_map.get(t1, 16)
        s2 = seed_map.get(t2, 16)
        if t1 not in seed_map or t2 not in seed_map:
            n_missing += 1
        lo, hi = min(s1, s2), max(s1, s2)
        key = (lo, hi)
        if key in seed_overrides:
            ov = seed_overrides[key]
            if s1 < s2:
                adj[idx] = max(adj[idx], ov)
            elif s2 < s1:
                adj[idx] = min(adj[idx], 1 - ov)
    n_changed = int((np.abs(adj - preds) > 1e-6).sum())
    print(f"          Teams missing from seed map : {n_missing}")
    print(f"          Seed overrides applied      : {n_changed} predictions adjusted")
    # Spot check Michigan(1236) vs Howard(1207)
    for idx, row_id in enumerate(ids):
        if row_id == "2026_1207_1236":
            print(f"          Michigan vs Howard check   : pred={adj[idx]:.4f} {chr(10236) if adj[idx]>=0.97 else chr(10236)} {"FIXED" if adj[idx]>=0.97 else "STILL WRONG"}")
            break
    return adj

def make_matchup_features(games, team_stats, recent, seed_feat,
                           massey_feat, elo_dict):
    df = games.copy()
    df["TeamA"] = df[["WTeamID","LTeamID"]].min(axis=1)
    df["TeamB"] = df[["WTeamID","LTeamID"]].max(axis=1)
    df["Label"] = (df["WTeamID"] == df["TeamA"]).astype(int)

    def mg(df, stats, prefix, side):
        cols = [c for c in stats.columns if c not in ["Season","TeamID"]]
        s    = stats.rename(columns={c: f"{prefix}_{c}" for c in cols})
        return df.merge(s, left_on=["Season", side],
                        right_on=["Season","TeamID"], how="left") \
                 .drop(columns=["TeamID"], errors="ignore")

    rf = recent[["Season","TeamID","RecentWinPct","RecentNetPts",
                 "RecentPtsFor","RecentPtsAgainst"]]
    df = mg(df, team_stats, "A", "TeamA")
    df = mg(df, team_stats, "B", "TeamB")
    df = mg(df, rf,         "A", "TeamA")
    df = mg(df, rf,         "B", "TeamB")
    df = mg(df, seed_feat,  "A", "TeamA")
    df = mg(df, seed_feat,  "B", "TeamB")
    if massey_feat is not None:
        mc = [c for c in massey_feat.columns if c not in ["Season","TeamID"]]
        df = mg(df, massey_feat[["Season","TeamID"]+mc], "A", "TeamA")
        df = mg(df, massey_feat[["Season","TeamID"]+mc], "B", "TeamB")

    df["A_ELO"] = df.apply(
        lambda r: elo_dict.get((r["Season"]-1, int(r["TeamA"])), ELO_BASE), axis=1)
    df["B_ELO"] = df.apply(
        lambda r: elo_dict.get((r["Season"]-1, int(r["TeamB"])), ELO_BASE), axis=1)

    feature_cols = []
    for ca in [c for c in df.columns if c.startswith("A_")]:
        base = ca[2:]
        cb   = f"B_{base}"
        if cb in df.columns:
            df[f"diff_{base}"] = df[ca] - df[cb]
            feature_cols.extend([f"diff_{base}", ca, cb])
    df["ELO_WinProb"] = 1 / (1 + 10 ** ((df["B_ELO"] - df["A_ELO"]) / 400))
    feature_cols = list(dict.fromkeys(feature_cols + ["ELO_WinProb"]))
    return df, feature_cols

def build_pred_features(pairs_df, stats_2025, recent_2025,
                         seed_pred, massey_2025, elo_final_adj, feature_cols):
    df = pairs_df.copy()

    def mg(df, ref, prefix, key):
        cols = [c for c in ref.columns if c != "TeamID"]
        renamed = ref.rename(columns={c: f"{prefix}_{c}" for c in cols})
        return df.merge(renamed, left_on=key, right_on="TeamID", how="left") \
                 .drop(columns=["TeamID"], errors="ignore")

    s  = stats_2025.drop(columns=["Season"])
    r  = recent_2025[["TeamID","RecentWinPct","RecentNetPts",
                       "RecentPtsFor","RecentPtsAgainst"]]
    sd = seed_pred   # FIX ① — 2026 seeds

    df = mg(df, s,  "A", "TeamA")
    df = mg(df, s,  "B", "TeamB")
    df = mg(df, r,  "A", "TeamA")
    df = mg(df, r,  "B", "TeamB")
    df = mg(df, sd, "A", "TeamA")
    df = mg(df, sd, "B", "TeamB")

    if massey_2025 is not None and len(massey_2025) > 0:
        mc = massey_2025.drop(columns=["Season"])
        df = mg(df, mc, "A", "TeamA")
        df = mg(df, mc, "B", "TeamB")

    # FIX ② — use conference-adjusted ELO
    df["A_ELO"] = df["TeamA"].apply(lambda t: elo_final_adj.get(int(t), ELO_BASE))
    df["B_ELO"] = df["TeamB"].apply(lambda t: elo_final_adj.get(int(t), ELO_BASE))

    for ca in [c for c in df.columns if c.startswith("A_")]:
        base = ca[2:]
        cb   = f"B_{base}"
        if cb in df.columns:
            df[f"diff_{base}"] = df[ca] - df[cb]

    df["ELO_WinProb"] = 1 / (1 + 10 ** ((df["B_ELO"] - df["A_ELO"]) / 400))
    return df.reindex(columns=feature_cols).fillna(0)

def train_and_predict(reg_df, tourney_df, seeds_df, massey_feat,
                      lgbm_params, xgb_params, pred_pairs,
                      conf_csv, seed_overrides, label):
    """Full pipeline for one gender."""

    print(f"\n{'█'*62}")
    print(f"  {label} PIPELINE")
    print(f"{'█'*62}")

    # STEP 2: ELO
    print(f"\n  STEP 2 | Computing ELO …")
    elo_dict, elo_final = compute_elo(reg_df, tourney_df)
    print(f"          {len(elo_dict):,} (season, team) ELO entries")

    # FIX ② — conference penalty
    print(f"  FIX ② | Applying conference strength penalty …")
    conf_penalty  = build_conf_penalty(conf_csv)
    elo_final_adj = apply_conf_penalty(elo_final, conf_penalty, season=2025)
    n_pen = sum(1 for v in conf_penalty.values() if v < 0)
    print(f"          {n_pen:,} team-seasons penalised")

    # STEP 3: Features
    print(f"  STEP 3 | Building team stats, recent form, seeds …")
    team_stats = build_team_stats(reg_df, tourney_df)
    recent     = build_recent_form(reg_df)

    seed_feat = seeds_df[seeds_df["Season"].isin(TRAIN_SEASONS)].copy()
    seed_feat["SeedNum"] = seed_feat["Seed"].apply(parse_seed_num)
    seed_feat = seed_feat[["Season","TeamID","SeedNum"]]
    print(f"          Team-season rows : {len(team_stats):,}")
    print(f"          Massey           : {'available' if massey_feat is not None else 'N/A'}")

    # STEP 4: Matchup features
    print(f"  STEP 4 | Building matchup feature matrix …")
    reg_df["IsTourney"]     = False
    tourney_df["IsTourney"] = True
    all_train = pd.concat([reg_df, tourney_df], ignore_index=True)

    train_df, feature_cols = make_matchup_features(
        all_train, team_stats, recent, seed_feat, massey_feat, elo_dict)
    X_tr = train_df[feature_cols].fillna(0)
    y_tr = train_df["Label"]
    print(f"          Train rows : {len(X_tr):,}  |  Features : {len(feature_cols)}")

    # STEP 5: Train models
    print(f"  STEP 5 | Training LightGBM …")
    lgbm_m = lgb.LGBMClassifier(**lgbm_params)
    lgbm_m.fit(X_tr, y_tr)
    print(f"          ✓ LightGBM done")

    split  = int(len(X_tr) * 0.9)
    X_trx  = X_tr.iloc[:split];  y_trx = y_tr.iloc[:split]
    X_valx = X_tr.iloc[split:];  y_valx = y_tr.iloc[split:]

    print(f"          Training XGBoost …")
    xgb_m = xgb.XGBClassifier(**xgb_params)
    xgb_m.fit(X_trx, y_trx, eval_set=[(X_valx, y_valx)], verbose=False)
    print(f"          ✓ XGBoost done")

    p_lv = lgbm_m.predict_proba(X_valx)[:,1]
    p_xv = xgb_m.predict_proba(X_valx)[:,1]
    best_w, best_ll = 0.5, 99
    for w in np.arange(0.05, 0.96, 0.05):
        ll = log_loss(y_valx, w*p_lv + (1-w)*p_xv)
        if ll < best_ll:
            best_ll, best_w = ll, w
    print(f"          Blend → LGBM: {best_w:.2f}  XGB: {1-best_w:.2f}  "
          f"val LogLoss={best_ll:.5f}")

    # 2025 tourney eval
    mask_eval = (train_df["Season"] == 2025) & (train_df["IsTourney"] == True)
    if mask_eval.sum() > 0:
        X_e = train_df.loc[mask_eval, feature_cols].fillna(0)
        y_e = train_df.loc[mask_eval, "Label"]
        p_e = best_w*lgbm_m.predict_proba(X_e)[:,1] + \
              (1-best_w)*xgb_m.predict_proba(X_e)[:,1]
        acc = accuracy_score(y_e, (p_e > 0.5).astype(int))
        nc  = int(((p_e > 0.5) == y_e.values).sum())
        print(f"          [2025 Tourney] Accuracy={acc:.4f} ({nc}/{len(y_e)})  "
              f"LogLoss={log_loss(y_e,p_e):.5f}  "
              f"AUC={roc_auc_score(y_e,p_e):.4f}")

    # STEP 6: 2026 prediction features
    print(f"\n  STEP 6 | Building 2026 prediction features …")
    stats_2025  = team_stats[team_stats["Season"] == 2025].copy()
    recent_2025 = recent[recent["Season"] == 2025].copy()
    massey_2025 = massey_feat[massey_feat["Season"] == 2025].copy() \
                  if massey_feat is not None else None

    # FIX ① — use 2026 seeds, fall back to 2025 if not available
    if 2026 in seeds_df["Season"].values:
        seed_pred = seeds_df[seeds_df["Season"] == 2026].copy()
        print(f"          ✅ FIX ① : Using 2026 tournament seeds")
    else:
        seed_pred = seeds_df[seeds_df["Season"] == 2025].copy()
        print(f"          ⚠️  FIX ① : 2026 seeds not found, falling back to 2025")
    seed_pred["SeedNum"] = seed_pred["Seed"].apply(parse_seed_num)
    seed_pred = seed_pred[["TeamID","SeedNum"]]

    # Build seed map for override lookup
    seed_map_2026 = dict(zip(seed_pred["TeamID"], seed_pred["SeedNum"]))
    print(f"          {len(seed_map_2026)} teams with 2026 seed assignments")
    print(f"          Stats 2025  : {len(stats_2025):,} teams")
    print(f"          Recent 2025 : {len(recent_2025):,} teams")
    print(f"          Pred pairs  : {len(pred_pairs):,}")

    X_pred = build_pred_features(
        pred_pairs, stats_2025, recent_2025,
        seed_pred, massey_2025, elo_final_adj, feature_cols)
    print(f"          Feature matrix : {X_pred.shape}")

    # STEP 7: Predict
    print(f"\n  STEP 7 | Predicting …")
    p_lgbm = lgbm_m.predict_proba(X_pred)[:,1]
    p_xgb  = xgb_m.predict_proba(X_pred)[:,1]
    preds  = best_w*p_lgbm + (1-best_w)*p_xgb

    # FIX ③ — seed overrides
    print(f"  FIX ③ | Applying seed overrides …")
    preds = apply_seed_overrides(
        preds, pred_pairs["ID"].values, seed_map_2026, seed_overrides)

    preds = np.clip(preds, 0.025, 0.975)
    print(f"          mean={preds.mean():.4f}  min={preds.min():.4f}  "
          f"max={preds.max():.4f}  std={preds.std():.4f}")

    # FIX ④ — ID dict mapping
    id_to_pred = dict(zip(pred_pairs["ID"].values, preds))
    print(f"          ✅ {label} done — {len(id_to_pred):,} predictions")
    return id_to_pred


# ─────────────────────────────────────────────────────────────
# STEP 1 — LOAD STAGE 2 SAMPLE & SPLIT BY GENDER
# ─────────────────────────────────────────────────────────────
print("\n" + "="*62)
print("  STEP 1 | Loading Stage 2 sample submission …")
print("="*62)

stage2 = pd.read_csv(DATA_DIR / "SampleSubmissionStage2.csv")
parts           = stage2["ID"].str.split("_", expand=True)
stage2["Season"] = parts[0].astype(int)
stage2["TeamA"]  = parts[1].astype(int)
stage2["TeamB"]  = parts[2].astype(int)

men_mask   = stage2["TeamA"] < 3000
women_mask = stage2["TeamA"] >= 3000
men_pairs   = stage2[men_mask][["ID","TeamA","TeamB"]].copy()   # FIX ④ keep ID
women_pairs = stage2[women_mask][["ID","TeamA","TeamB"]].copy() # FIX ④ keep ID

print(f"  Total rows   : {len(stage2):,}")
print(f"  Men's rows   : {len(men_pairs):,}  (TeamID 1xxx)")
print(f"  Women's rows : {len(women_pairs):,}  (TeamID 3xxx)")

# Load men's data
print("\n  Loading Men's raw data …")
m_reg     = pd.read_csv(DATA_DIR / "MRegularSeasonDetailedResults.csv")
m_tourney = pd.read_csv(DATA_DIR / "MNCAATourneyDetailedResults.csv")
m_seeds   = pd.read_csv(DATA_DIR / "MNCAATourneySeeds.csv")
m_massey  = build_massey(TRAIN_SEASONS)
m_reg     = m_reg[m_reg["Season"].isin(TRAIN_SEASONS)].copy()
m_tourney = m_tourney[m_tourney["Season"].isin(TRAIN_SEASONS)].copy()
print(f"  M Regular: {len(m_reg):,}  |  M Tourney: {len(m_tourney):,}  |  "
      f"Massey: {'yes' if m_massey is not None else 'no'}")

# Load women's data
print("  Loading Women's raw data …")
w_reg     = pd.read_csv(DATA_DIR / "WRegularSeasonDetailedResults.csv")
w_tourney = pd.read_csv(DATA_DIR / "WNCAATourneyDetailedResults.csv")
w_seeds   = pd.read_csv(DATA_DIR / "WNCAATourneySeeds.csv")
w_reg     = w_reg[w_reg["Season"].isin(TRAIN_SEASONS)].copy()
w_tourney = w_tourney[w_tourney["Season"].isin(TRAIN_SEASONS)].copy()
print(f"  W Regular: {len(w_reg):,}  |  W Tourney: {len(w_tourney):,}  |  Massey: N/A")

# ─────────────────────────────────────────────────────────────
# RUN BOTH PIPELINES
# ─────────────────────────────────────────────────────────────
mens_preds = train_and_predict(
    m_reg, m_tourney, m_seeds, m_massey,
    MEN_LGBM, MEN_XGB,
    men_pairs,
    conf_csv       = "MTeamConferences.csv",
    seed_overrides = SEED_OVERRIDES_MENS,
    label          = "🏀 MEN'S",
)

womens_preds = train_and_predict(
    w_reg, w_tourney, w_seeds, None,
    WOMEN_LGBM, WOMEN_XGB,
    women_pairs,
    conf_csv       = "WTeamConferences.csv",
    seed_overrides = SEED_OVERRIDES_WOMENS,
    label          = "🏀 WOMEN'S",
)

# ─────────────────────────────────────────────────────────────
# STEP 8 — COMBINE, SAVE, SANITY CHECK
# ─────────────────────────────────────────────────────────────
print("\n" + "="*62)
print("  STEP 8 | Combining and saving submission …")
print("="*62)

# FIX ④ — map by ID dict (zero reindex drift)
all_preds      = {**mens_preds, **womens_preds}
stage2["Pred"] = stage2["ID"].map(all_preds)

n_missing = stage2["Pred"].isna().sum()
if n_missing:
    print(f"  ⚠️  {n_missing} IDs unmatched — filling with 0.5")
    stage2["Pred"] = stage2["Pred"].fillna(0.5)

stage2["Pred"] = stage2["Pred"].clip(0.025, 0.975).round(6)
submission = stage2[["ID","Pred"]]
submission.to_csv(OUTPUT, index=False)

# Sanity check
print(f"\n  {'─'*52}")
print(f"  File         : {OUTPUT}")
print(f"  Total rows   : {len(submission):,}  (expected 132,133) "
      f"{'✅' if len(submission)==132133 else '❌'}")
print(f"  Men's rows   : {men_mask.sum():,}")
print(f"  Women's rows : {women_mask.sum():,}")
print(f"  Any NaN      : {submission['Pred'].isna().sum()} "
      f"{'✅' if submission['Pred'].isna().sum()==0 else '❌'}")
print(f"  Pred min     : {submission['Pred'].min():.4f}  (floor=0.025)")
print(f"  Pred max     : {submission['Pred'].max():.4f}  (cap=0.975)")
print(f"  Pred mean    : {submission['Pred'].mean():.4f}  (should be ~0.5)")
print(f"  Pred std     : {submission['Pred'].std():.4f}  (>0.15 = good)")
print(f"  Unique vals  : {submission['Pred'].nunique():,}")
print(f"  {'─'*52}")
print(f"\n  First 5 rows:")
print(submission.head(5).to_string(index=False))
print(f"\n  Last 5 rows (women's):")
print(submission.tail(5).to_string(index=False))
print(f"\n✅  shaurya_final_1.csv ready — upload to Kaggle!")
print(f"\n  FIXES APPLIED:")
print(f"  ① 2026 seeds used for prediction features")
print(f"  ② Conference strength penalty (MEAC/SWAC = -150 ELO)")
print(f"  ③ Seed overrides: 1 vs 16 → min 97.5% for 1-seed")
print(f"  ④ ID dict mapping — no reindex drift")

🖥️  Using 8 CPU cores

  STEP 1 | Loading Stage 2 sample submission …
  Total rows   : 132,133
  Men's rows   : 66,430  (TeamID 1xxx)
  Women's rows : 65,703  (TeamID 3xxx)

  Loading Men's raw data …
  M Regular: 84,808  |  M Tourney: 1,001  |  Massey: yes
  Loading Women's raw data …
  W Regular: 81,708  |  W Tourney: 961  |  Massey: N/A

██████████████████████████████████████████████████████████████
  🏀 MEN'S PIPELINE
██████████████████████████████████████████████████████████████

  STEP 2 | Computing ELO …
          5,690 (season, team) ELO entries
  FIX ② | Applying conference strength penalty …
          3,766 team-seasons penalised
  STEP 3 | Building team stats, recent form, seeds …
          Team-season rows : 5,639
          Massey           : available
  STEP 4 | Building matchup feature matrix …
          Train rows : 85,809  |  Features : 172
  STEP 5 | Training LightGBM …
          ✓ LightGBM done
          Training XGBoost …
          ✓ XGBoost done
          Blend → LGB

In [2]:
# DIRECT TEST — paste and run this in Jupyter
import pandas as pd
import numpy as np

path = "/Users/shaurya/Downloads/shaurya_final_1.csv"
df = pd.read_csv(path)
parts = df['ID'].str.split('_', expand=True).astype(int)
df['T1'] = parts[1]; df['T2'] = parts[2]

row = df[(df['T1']==1207) & (df['T2']==1236)]
pred = row['Pred'].values[0]
p_mich = 1 - pred

print(f"Michigan vs Howard: P(Michigan) = {p_mich:.4f}")
print("Fix working:", "✅ YES" if p_mich >= 0.97 else "❌ NO")

Michigan vs Howard: P(Michigan) = 0.3003
Fix working: ❌ NO


In [5]:
import subprocess
result = subprocess.run(
    ['python3', '/Users/shaurya/Downloads/Updated_submission_stage2_final_one.py'],
    capture_output=False


_IncompleteInputError: incomplete input (1213722398.py, line 4)

In [6]:
"""
=============================================================================
NCAA March Mania 2026 — FINAL SUBMISSION GENERATOR (FULLY FIXED)
=============================================================================
Produces Updated_submission_stage2_final_one.csv with exactly 132,133 rows.
Men   (TeamID 1xxx) : trained on M files, men's best BO params
Women (TeamID 3xxx) : trained on W files, women's best BO params

FIXES APPLIED:
  ① 2026 seeds used for prediction (falls back to 2025 if not available)
  ② Conference strength penalty — weak conferences get ELO penalty
  ③ Seed overrides — 1 vs 16 can never be < 97.5% for 1-seed
  ④ ID dict mapping — no reindex drift
=============================================================================
"""

import warnings; warnings.filterwarnings("ignore")
import os, multiprocessing, re
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.metrics import log_loss, accuracy_score, roc_auc_score
import lightgbm as lgb
import xgboost as xgb

# ─────────────────────────────────────────────────────────────
# CONFIG
# ─────────────────────────────────────────────────────────────
DATA_DIR = Path("/Users/shaurya/Downloads")
OUTPUT   = DATA_DIR / "shaurya_final_1.csv"  # FIX ④ unique name

N_JOBS = multiprocessing.cpu_count()
os.environ["OMP_NUM_THREADS"]      = str(N_JOBS)
os.environ["MKL_NUM_THREADS"]      = str(N_JOBS)
os.environ["OPENBLAS_NUM_THREADS"] = str(N_JOBS)
print(f"🖥️  Using {N_JOBS} CPU cores")

TRAIN_SEASONS = list(range(2010, 2026))   # 2010–2025 inclusive
ELO_K         = 20
ELO_BASE      = 1500
ELO_HOME_ADV  = 100
SEED          = 42
np.random.seed(SEED)

STAT_COLS = ["FGM","FGA","FGM3","FGA3","FTM","FTA",
             "OR","DR","Ast","TO","Stl","Blk","PF"]

# ─────────────────────────────────────────────────────────────
# FIX ③ — SEED OVERRIDES
# Forces extreme seed matchups to realistic probabilities.
# ─────────────────────────────────────────────────────────────
SEED_OVERRIDES_MENS = {
    (1, 16): 0.975,
    (1, 15): 0.945,
    (1, 14): 0.920,
    (1, 13): 0.900,
    (2, 15): 0.930,
    (2, 16): 0.960,
    (3, 14): 0.850,
    (4, 13): 0.820,
    (5, 12): 0.650,
    (6, 11): 0.620,
    (7, 10): 0.600,
    (8,  9): 0.520,
}
SEED_OVERRIDES_WOMENS = {
    (1, 16): 0.985,
    (1, 15): 0.965,
    (1, 14): 0.945,
    (1, 13): 0.925,
    (2, 15): 0.955,
    (2, 16): 0.975,
    (3, 14): 0.875,
    (4, 13): 0.850,
    (5, 12): 0.720,
    (6, 11): 0.680,
    (7, 10): 0.650,
    (8,  9): 0.540,
}

# ─────────────────────────────────────────────────────────────
# FIX ② — CONFERENCE STRENGTH TIERS
# Weak-conference teams have inflated stats vs weak opponents.
# ─────────────────────────────────────────────────────────────
POWER_CONFS = {
    "acc","big_east","big_ten","big_twelve","pac_twelve",
    "sec","aac","mwc","wcc","a_ten"
}
WEAK_CONFS = {
    "meac","swac","nec","big_south","independent"
}
CONF_ELO_PENALTY = {
    "power"    :    0,
    "mid_major":  -50,
    "weak"     : -150,
}

# ─────────────────────────────────────────────────────────────
# BEST PARAMS — MEN'S
# ─────────────────────────────────────────────────────────────
MEN_LGBM = {
    "num_leaves"        : 170,
    "max_depth"         : 12,
    "learning_rate"     : 0.005,
    "n_estimators"      : 1308 + 200,
    "min_child_samples" : 5,
    "subsample"         : 0.5,
    "colsample_bytree"  : 0.5,
    "reg_alpha"         : 0.0,
    "reg_lambda"        : 0.0,
    "objective"         : "binary",
    "metric"            : "binary_logloss",
    "random_state"      : SEED,
    "n_jobs"            : N_JOBS,
    "device_type"       : "cpu",
    "num_threads"       : N_JOBS,
    "verbose"           : -1,
}
MEN_XGB = {
    "n_estimators"          : 1638 + 200,
    "max_depth"             : 10,
    "learning_rate"         : 0.0255778,
    "min_child_weight"      : 2.84511,
    "subsample"             : 0.848826,
    "colsample_bytree"      : 0.562814,
    "gamma"                 : 1.59748,
    "reg_alpha"             : 4.86392,
    "reg_lambda"            : 1.04622,
    "early_stopping_rounds" : 50,
    "objective"             : "binary:logistic",
    "eval_metric"           : "logloss",
    "random_state"          : SEED,
    "n_jobs"                : N_JOBS,
    "verbosity"             : 0,
    "tree_method"           : "hist",
}

# ─────────────────────────────────────────────────────────────
# BEST PARAMS — WOMEN'S
# ─────────────────────────────────────────────────────────────
WOMEN_LGBM = {
    "num_leaves"        : 164,
    "max_depth"         : 8,
    "learning_rate"     : 0.0187029,
    "n_estimators"      : 1294 + 200,
    "min_child_samples" : 22,
    "subsample"         : 0.532526,
    "colsample_bytree"  : 0.974443,
    "reg_alpha"         : 4.82816,
    "reg_lambda"        : 4.04199,
    "objective"         : "binary",
    "metric"            : "binary_logloss",
    "random_state"      : SEED,
    "n_jobs"            : N_JOBS,
    "device_type"       : "cpu",
    "num_threads"       : N_JOBS,
    "verbose"           : -1,
}
WOMEN_XGB = {
    "n_estimators"          : 1898 + 200,
    "max_depth"             : 7,
    "learning_rate"         : 0.0559337,
    "min_child_weight"      : 18.9736,
    "subsample"             : 0.842968,
    "colsample_bytree"      : 0.895046,
    "gamma"                 : 0.171643,
    "reg_alpha"             : 1.13418,
    "reg_lambda"            : 2.44057,
    "early_stopping_rounds" : 50,
    "objective"             : "binary:logistic",
    "eval_metric"           : "logloss",
    "random_state"          : SEED,
    "n_jobs"                : N_JOBS,
    "verbosity"             : 0,
    "tree_method"           : "hist",
}

# ─────────────────────────────────────────────────────────────
# SHARED HELPER FUNCTIONS
# ─────────────────────────────────────────────────────────────
def parse_seed_num(s):
    if pd.isna(s): return 16
    m = re.search(r"\d+", str(s))
    return int(m.group()) if m else 16

def compute_elo(reg_df, tourney_df):
    cols = ["Season","DayNum","WTeamID","LTeamID","WLoc","WScore","LScore"]
    all_games = pd.concat([reg_df[cols], tourney_df[cols]]) \
                  .sort_values(["Season","DayNum"]).reset_index(drop=True)
    elo = {}
    season_end = {}
    for season in sorted(all_games["Season"].unique()):
        for tid in list(elo):
            elo[tid] = ELO_BASE * 0.25 + elo[tid] * 0.75
        for _, r in all_games[all_games["Season"] == season].iterrows():
            w, l   = int(r["WTeamID"]), int(r["LTeamID"])
            rw, rl = elo.get(w, ELO_BASE), elo.get(l, ELO_BASE)
            loc    = r["WLoc"]
            rw_adj = rw + ELO_HOME_ADV if loc=="H" \
                     else (rw - ELO_HOME_ADV if loc=="A" else rw)
            mov    = min(abs(int(r["WScore"]) - int(r["LScore"])), 20)
            mov_m  = np.log1p(mov) * (2.2 / ((rw_adj - rl) * 0.001 + 2.2))
            exp_w  = 1 / (1 + 10 ** ((rl - rw_adj) / 400))
            delta  = ELO_K * mov_m * (1 - exp_w)
            elo[w] = rw + delta
            elo[l] = rl - delta
        for tid, rat in elo.items():
            season_end[(season, tid)] = rat
    return season_end, elo

def explode_games(games):
    rw = {"WTeamID":"TeamID","LTeamID":"OppID",
          "WScore":"PtsFor","LScore":"PtsAgainst",
          **{f"W{s}":s for s in STAT_COLS},
          **{f"L{s}":f"Opp{s}" for s in STAT_COLS}}
    rl = {"LTeamID":"TeamID","WTeamID":"OppID",
          "LScore":"PtsFor","WScore":"PtsAgainst",
          **{f"L{s}":s for s in STAT_COLS},
          **{f"W{s}":f"Opp{s}" for s in STAT_COLS}}
    keep = ["Season","DayNum","TeamID","OppID","PtsFor","PtsAgainst"] + \
           STAT_COLS + [f"Opp{s}" for s in STAT_COLS]
    w = games.rename(columns=rw); w["Win"] = 1
    l = games.rename(columns=rl); l["Win"] = 0
    return pd.concat([w[keep+["Win"]], l[keep+["Win"]]], ignore_index=True)

def build_team_stats(reg_df, tourney_df):
    tg  = explode_games(pd.concat([reg_df, tourney_df]))
    eps = 1e-6
    agg = tg.groupby(["Season","TeamID"]).agg(
        GP=("Win","count"), Wins=("Win","sum"),
        PtsFor=("PtsFor","mean"), PtsAgainst=("PtsAgainst","mean"),
        FGM=("FGM","mean"), FGA=("FGA","mean"),
        FGM3=("FGM3","mean"), FGA3=("FGA3","mean"),
        FTM=("FTM","mean"), FTA=("FTA","mean"),
        OR=("OR","mean"),   DR=("DR","mean"),
        Ast=("Ast","mean"), TO=("TO","mean"),
        Stl=("Stl","mean"), Blk=("Blk","mean"), PF=("PF","mean"),
        OppFGM=("OppFGM","mean"),   OppFGA=("OppFGA","mean"),
        OppFGM3=("OppFGM3","mean"), OppFGA3=("OppFGA3","mean"),
        OppOR=("OppOR","mean"),     OppDR=("OppDR","mean"),
    ).reset_index()
    agg["WinPct"]         = agg["Wins"]    / (agg["GP"]      + eps)
    agg["FGPct"]          = agg["FGM"]     / (agg["FGA"]     + eps)
    agg["FG3Pct"]         = agg["FGM3"]    / (agg["FGA3"]    + eps)
    agg["FTPct"]          = agg["FTM"]     / (agg["FTA"]     + eps)
    agg["OppFGPct"]       = agg["OppFGM"]  / (agg["OppFGA"]  + eps)
    agg["eFGPct"]         = (agg["FGM"]    + 0.5*agg["FGM3"])    / (agg["FGA"]    + eps)
    agg["OppEFGPct"]      = (agg["OppFGM"] + 0.5*agg["OppFGM3"]) / (agg["OppFGA"] + eps)
    poss                  = agg["FGA"] - agg["OR"] + agg["TO"] + 0.44*agg["FTA"]
    agg["TOVRate"]        = agg["TO"]  / (poss              + eps)
    agg["ORBRate"]        = agg["OR"]  / (agg["OR"] + agg["OppDR"] + eps)
    agg["FTRate"]         = agg["FTM"] / (agg["FGA"]        + eps)
    agg["DRBRate"]        = agg["DR"]  / (agg["DR"] + agg["OppOR"] + eps)
    agg["Pace"]           = poss       / (agg["GP"]          + eps)
    agg["NetRtg"]         = agg["PtsFor"] - agg["PtsAgainst"]
    agg["OffRtg"]         = agg["PtsFor"]     / (poss + eps) * 100
    agg["DefRtg"]         = agg["PtsAgainst"] / (poss + eps) * 100
    agg["NetRtg2"]        = agg["OffRtg"] - agg["DefRtg"]
    agg["AstTO"]          = agg["Ast"] / (agg["TO"]  + eps)
    agg["StlBlk"]         = (agg["Stl"] + agg["Blk"]) / (agg["OppFGA"] + eps)
    agg["ThreePtRate"]    = agg["FGA3"]    / (agg["FGA"]    + eps)
    agg["OppThreePtRate"] = agg["OppFGA3"] / (agg["OppFGA"] + eps)
    return agg

def build_recent_form(reg_df, n=10):
    tg  = explode_games(reg_df).sort_values(["Season","TeamID","DayNum"])
    rec = tg.groupby(["Season","TeamID"]).tail(n)
    rf  = rec.groupby(["Season","TeamID"]).agg(
        RecentWinPct=("Win","mean"),
        RecentPtsFor=("PtsFor","mean"),
        RecentPtsAgainst=("PtsAgainst","mean"),
    ).reset_index()
    rf["RecentNetPts"] = rf["RecentPtsFor"] - rf["RecentPtsAgainst"]
    return rf

def build_massey(seasons):
    fp = DATA_DIR / "MMasseyOrdinals.csv"
    if not fp.exists(): return None
    df = pd.read_csv(fp)
    df = df[(df["Season"].isin(seasons)) & (df["RankingDayNum"] <= 128)]
    result = (df.groupby(["Season","TeamID"])["OrdinalRank"].mean()
                .reset_index().rename(columns={"OrdinalRank":"AvgRank"}))
    for sys in ["SAG","POM","WOL","MOR","DOK","REW","NET"]:
        sub = df[df["SystemName"] == sys]
        if len(sub) == 0: continue
        s = (sub.groupby(["Season","TeamID"])["OrdinalRank"].mean()
                .reset_index().rename(columns={"OrdinalRank":f"Rank_{sys}"}))
        result = result.merge(s, on=["Season","TeamID"], how="left")
    return result

# FIX ② — build conference penalty dict
def build_conf_penalty(conf_csv):
    fp = DATA_DIR / conf_csv
    if not fp.exists():
        print(f"  ⚠️  {conf_csv} not found — skipping conference penalty")
        return {}
    df = pd.read_csv(fp)
    df = df[df["Season"].isin(TRAIN_SEASONS)]
    penalty = {}
    for _, r in df.iterrows():
        conf = str(r["ConfAbbrev"]).lower()
        if conf in POWER_CONFS:
            p = CONF_ELO_PENALTY["power"]
        elif conf in WEAK_CONFS:
            p = CONF_ELO_PENALTY["weak"]
        else:
            p = CONF_ELO_PENALTY["mid_major"]
        penalty[(int(r["Season"]), int(r["TeamID"]))] = p
    return penalty

def apply_conf_penalty(elo_final, conf_penalty, season=2025):
    adjusted = {}
    for tid, rating in elo_final.items():
        pen = conf_penalty.get((season, tid), CONF_ELO_PENALTY["mid_major"])
        adjusted[tid] = rating + pen
    return adjusted

# FIX ③ — apply seed overrides
def apply_seed_overrides(preds, ids, seed_map, seed_overrides):
    adj = preds.copy()
    print(f"          Seed map has {len(seed_map)} teams")
    n_missing = 0
    for idx, row_id in enumerate(ids):
        parts = row_id.split("_")
        t1, t2 = int(parts[1]), int(parts[2])
        # Default missing teams to 16 (not 8) so 1-seeds always trigger override
        s1 = seed_map.get(t1, 16)
        s2 = seed_map.get(t2, 16)
        if t1 not in seed_map or t2 not in seed_map:
            n_missing += 1
        lo, hi = min(s1, s2), max(s1, s2)
        key = (lo, hi)
        if key in seed_overrides:
            ov = seed_overrides[key]
            if s1 < s2:
                adj[idx] = max(adj[idx], ov)
            elif s2 < s1:
                adj[idx] = min(adj[idx], 1 - ov)
    n_changed = int((np.abs(adj - preds) > 1e-6).sum())
    print(f"          Teams missing from seed map : {n_missing}")
    print(f"          Seed overrides applied      : {n_changed} predictions adjusted")
    # Spot check Michigan(1236) vs Howard(1207)
    for idx, row_id in enumerate(ids):
        if row_id == "2026_1207_1236":
            status = "FIXED" if adj[idx] >= 0.97 else "STILL WRONG"
            print(f"          Michigan vs Howard check   : pred={adj[idx]:.4f}  [{status}]")
            break
    return adj

def make_matchup_features(games, team_stats, recent, seed_feat,
                           massey_feat, elo_dict):
    df = games.copy()
    df["TeamA"] = df[["WTeamID","LTeamID"]].min(axis=1)
    df["TeamB"] = df[["WTeamID","LTeamID"]].max(axis=1)
    df["Label"] = (df["WTeamID"] == df["TeamA"]).astype(int)

    def mg(df, stats, prefix, side):
        cols = [c for c in stats.columns if c not in ["Season","TeamID"]]
        s    = stats.rename(columns={c: f"{prefix}_{c}" for c in cols})
        return df.merge(s, left_on=["Season", side],
                        right_on=["Season","TeamID"], how="left") \
                 .drop(columns=["TeamID"], errors="ignore")

    rf = recent[["Season","TeamID","RecentWinPct","RecentNetPts",
                 "RecentPtsFor","RecentPtsAgainst"]]
    df = mg(df, team_stats, "A", "TeamA")
    df = mg(df, team_stats, "B", "TeamB")
    df = mg(df, rf,         "A", "TeamA")
    df = mg(df, rf,         "B", "TeamB")
    df = mg(df, seed_feat,  "A", "TeamA")
    df = mg(df, seed_feat,  "B", "TeamB")
    if massey_feat is not None:
        mc = [c for c in massey_feat.columns if c not in ["Season","TeamID"]]
        df = mg(df, massey_feat[["Season","TeamID"]+mc], "A", "TeamA")
        df = mg(df, massey_feat[["Season","TeamID"]+mc], "B", "TeamB")

    df["A_ELO"] = df.apply(
        lambda r: elo_dict.get((r["Season"]-1, int(r["TeamA"])), ELO_BASE), axis=1)
    df["B_ELO"] = df.apply(
        lambda r: elo_dict.get((r["Season"]-1, int(r["TeamB"])), ELO_BASE), axis=1)

    feature_cols = []
    for ca in [c for c in df.columns if c.startswith("A_")]:
        base = ca[2:]
        cb   = f"B_{base}"
        if cb in df.columns:
            df[f"diff_{base}"] = df[ca] - df[cb]
            feature_cols.extend([f"diff_{base}", ca, cb])
    df["ELO_WinProb"] = 1 / (1 + 10 ** ((df["B_ELO"] - df["A_ELO"]) / 400))
    feature_cols = list(dict.fromkeys(feature_cols + ["ELO_WinProb"]))
    return df, feature_cols

def build_pred_features(pairs_df, stats_2025, recent_2025,
                         seed_pred, massey_2025, elo_final_adj, feature_cols):
    df = pairs_df.copy()

    def mg(df, ref, prefix, key):
        cols = [c for c in ref.columns if c != "TeamID"]
        renamed = ref.rename(columns={c: f"{prefix}_{c}" for c in cols})
        return df.merge(renamed, left_on=key, right_on="TeamID", how="left") \
                 .drop(columns=["TeamID"], errors="ignore")

    s  = stats_2025.drop(columns=["Season"])
    r  = recent_2025[["TeamID","RecentWinPct","RecentNetPts",
                       "RecentPtsFor","RecentPtsAgainst"]]
    sd = seed_pred   # FIX ① — 2026 seeds

    df = mg(df, s,  "A", "TeamA")
    df = mg(df, s,  "B", "TeamB")
    df = mg(df, r,  "A", "TeamA")
    df = mg(df, r,  "B", "TeamB")
    df = mg(df, sd, "A", "TeamA")
    df = mg(df, sd, "B", "TeamB")

    if massey_2025 is not None and len(massey_2025) > 0:
        mc = massey_2025.drop(columns=["Season"])
        df = mg(df, mc, "A", "TeamA")
        df = mg(df, mc, "B", "TeamB")

    # FIX ② — use conference-adjusted ELO
    df["A_ELO"] = df["TeamA"].apply(lambda t: elo_final_adj.get(int(t), ELO_BASE))
    df["B_ELO"] = df["TeamB"].apply(lambda t: elo_final_adj.get(int(t), ELO_BASE))

    for ca in [c for c in df.columns if c.startswith("A_")]:
        base = ca[2:]
        cb   = f"B_{base}"
        if cb in df.columns:
            df[f"diff_{base}"] = df[ca] - df[cb]

    df["ELO_WinProb"] = 1 / (1 + 10 ** ((df["B_ELO"] - df["A_ELO"]) / 400))
    return df.reindex(columns=feature_cols).fillna(0)

def train_and_predict(reg_df, tourney_df, seeds_df, massey_feat,
                      lgbm_params, xgb_params, pred_pairs,
                      conf_csv, seed_overrides, label):
    """Full pipeline for one gender."""

    print(f"\n{'█'*62}")
    print(f"  {label} PIPELINE")
    print(f"{'█'*62}")

    # STEP 2: ELO
    print(f"\n  STEP 2 | Computing ELO …")
    elo_dict, elo_final = compute_elo(reg_df, tourney_df)
    print(f"          {len(elo_dict):,} (season, team) ELO entries")

    # FIX ② — conference penalty
    print(f"  FIX ② | Applying conference strength penalty …")
    conf_penalty  = build_conf_penalty(conf_csv)
    elo_final_adj = apply_conf_penalty(elo_final, conf_penalty, season=2025)
    n_pen = sum(1 for v in conf_penalty.values() if v < 0)
    print(f"          {n_pen:,} team-seasons penalised")

    # STEP 3: Features
    print(f"  STEP 3 | Building team stats, recent form, seeds …")
    team_stats = build_team_stats(reg_df, tourney_df)
    recent     = build_recent_form(reg_df)

    seed_feat = seeds_df[seeds_df["Season"].isin(TRAIN_SEASONS)].copy()
    seed_feat["SeedNum"] = seed_feat["Seed"].apply(parse_seed_num)
    seed_feat = seed_feat[["Season","TeamID","SeedNum"]]
    print(f"          Team-season rows : {len(team_stats):,}")
    print(f"          Massey           : {'available' if massey_feat is not None else 'N/A'}")

    # STEP 4: Matchup features
    print(f"  STEP 4 | Building matchup feature matrix …")
    reg_df["IsTourney"]     = False
    tourney_df["IsTourney"] = True
    all_train = pd.concat([reg_df, tourney_df], ignore_index=True)

    train_df, feature_cols = make_matchup_features(
        all_train, team_stats, recent, seed_feat, massey_feat, elo_dict)
    X_tr = train_df[feature_cols].fillna(0)
    y_tr = train_df["Label"]
    print(f"          Train rows : {len(X_tr):,}  |  Features : {len(feature_cols)}")

    # STEP 5: Train models
    print(f"  STEP 5 | Training LightGBM …")
    lgbm_m = lgb.LGBMClassifier(**lgbm_params)
    lgbm_m.fit(X_tr, y_tr)
    print(f"          ✓ LightGBM done")

    split  = int(len(X_tr) * 0.9)
    X_trx  = X_tr.iloc[:split];  y_trx = y_tr.iloc[:split]
    X_valx = X_tr.iloc[split:];  y_valx = y_tr.iloc[split:]

    print(f"          Training XGBoost …")
    xgb_m = xgb.XGBClassifier(**xgb_params)
    xgb_m.fit(X_trx, y_trx, eval_set=[(X_valx, y_valx)], verbose=False)
    print(f"          ✓ XGBoost done")

    p_lv = lgbm_m.predict_proba(X_valx)[:,1]
    p_xv = xgb_m.predict_proba(X_valx)[:,1]
    best_w, best_ll = 0.5, 99
    for w in np.arange(0.05, 0.96, 0.05):
        ll = log_loss(y_valx, w*p_lv + (1-w)*p_xv)
        if ll < best_ll:
            best_ll, best_w = ll, w
    print(f"          Blend → LGBM: {best_w:.2f}  XGB: {1-best_w:.2f}  "
          f"val LogLoss={best_ll:.5f}")

    # 2025 tourney eval
    mask_eval = (train_df["Season"] == 2025) & (train_df["IsTourney"] == True)
    if mask_eval.sum() > 0:
        X_e = train_df.loc[mask_eval, feature_cols].fillna(0)
        y_e = train_df.loc[mask_eval, "Label"]
        p_e = best_w*lgbm_m.predict_proba(X_e)[:,1] + \
              (1-best_w)*xgb_m.predict_proba(X_e)[:,1]
        acc = accuracy_score(y_e, (p_e > 0.5).astype(int))
        nc  = int(((p_e > 0.5) == y_e.values).sum())
        print(f"          [2025 Tourney] Accuracy={acc:.4f} ({nc}/{len(y_e)})  "
              f"LogLoss={log_loss(y_e,p_e):.5f}  "
              f"AUC={roc_auc_score(y_e,p_e):.4f}")

    # STEP 6: 2026 prediction features
    print(f"\n  STEP 6 | Building 2026 prediction features …")
    stats_2025  = team_stats[team_stats["Season"] == 2025].copy()
    recent_2025 = recent[recent["Season"] == 2025].copy()
    massey_2025 = massey_feat[massey_feat["Season"] == 2025].copy() \
                  if massey_feat is not None else None

    # FIX ① — use 2026 seeds, fall back to 2025 if not available
    if 2026 in seeds_df["Season"].values:
        seed_pred = seeds_df[seeds_df["Season"] == 2026].copy()
        print(f"          ✅ FIX ① : Using 2026 tournament seeds")
    else:
        seed_pred = seeds_df[seeds_df["Season"] == 2025].copy()
        print(f"          ⚠️  FIX ① : 2026 seeds not found, falling back to 2025")
    seed_pred["SeedNum"] = seed_pred["Seed"].apply(parse_seed_num)
    seed_pred = seed_pred[["TeamID","SeedNum"]]

    # Build seed map for override lookup
    seed_map_2026 = dict(zip(seed_pred["TeamID"], seed_pred["SeedNum"]))
    print(f"          {len(seed_map_2026)} teams with 2026 seed assignments")
    print(f"          Stats 2025  : {len(stats_2025):,} teams")
    print(f"          Recent 2025 : {len(recent_2025):,} teams")
    print(f"          Pred pairs  : {len(pred_pairs):,}")

    X_pred = build_pred_features(
        pred_pairs, stats_2025, recent_2025,
        seed_pred, massey_2025, elo_final_adj, feature_cols)
    print(f"          Feature matrix : {X_pred.shape}")

    # STEP 7: Predict
    print(f"\n  STEP 7 | Predicting …")
    p_lgbm = lgbm_m.predict_proba(X_pred)[:,1]
    p_xgb  = xgb_m.predict_proba(X_pred)[:,1]
    preds  = best_w*p_lgbm + (1-best_w)*p_xgb

    # FIX ③ — seed overrides
    print(f"  FIX ③ | Applying seed overrides …")
    preds = apply_seed_overrides(
        preds, pred_pairs["ID"].values, seed_map_2026, seed_overrides)

    preds = np.clip(preds, 0.025, 0.975)
    print(f"          mean={preds.mean():.4f}  min={preds.min():.4f}  "
          f"max={preds.max():.4f}  std={preds.std():.4f}")

    # FIX ④ — ID dict mapping
    id_to_pred = dict(zip(pred_pairs["ID"].values, preds))
    print(f"          ✅ {label} done — {len(id_to_pred):,} predictions")
    return id_to_pred


# ─────────────────────────────────────────────────────────────
# STEP 1 — LOAD STAGE 2 SAMPLE & SPLIT BY GENDER
# ─────────────────────────────────────────────────────────────
print("\n" + "="*62)
print("  STEP 1 | Loading Stage 2 sample submission …")
print("="*62)

stage2 = pd.read_csv(DATA_DIR / "SampleSubmissionStage2.csv")
parts           = stage2["ID"].str.split("_", expand=True)
stage2["Season"] = parts[0].astype(int)
stage2["TeamA"]  = parts[1].astype(int)
stage2["TeamB"]  = parts[2].astype(int)

men_mask   = stage2["TeamA"] < 3000
women_mask = stage2["TeamA"] >= 3000
men_pairs   = stage2[men_mask][["ID","TeamA","TeamB"]].copy()   # FIX ④ keep ID
women_pairs = stage2[women_mask][["ID","TeamA","TeamB"]].copy() # FIX ④ keep ID

print(f"  Total rows   : {len(stage2):,}")
print(f"  Men's rows   : {len(men_pairs):,}  (TeamID 1xxx)")
print(f"  Women's rows : {len(women_pairs):,}  (TeamID 3xxx)")

# Load men's data
print("\n  Loading Men's raw data …")
m_reg     = pd.read_csv(DATA_DIR / "MRegularSeasonDetailedResults.csv")
m_tourney = pd.read_csv(DATA_DIR / "MNCAATourneyDetailedResults.csv")
m_seeds   = pd.read_csv(DATA_DIR / "MNCAATourneySeeds.csv")
m_massey  = build_massey(TRAIN_SEASONS)
m_reg     = m_reg[m_reg["Season"].isin(TRAIN_SEASONS)].copy()
m_tourney = m_tourney[m_tourney["Season"].isin(TRAIN_SEASONS)].copy()
print(f"  M Regular: {len(m_reg):,}  |  M Tourney: {len(m_tourney):,}  |  "
      f"Massey: {'yes' if m_massey is not None else 'no'}")

# Load women's data
print("  Loading Women's raw data …")
w_reg     = pd.read_csv(DATA_DIR / "WRegularSeasonDetailedResults.csv")
w_tourney = pd.read_csv(DATA_DIR / "WNCAATourneyDetailedResults.csv")
w_seeds   = pd.read_csv(DATA_DIR / "WNCAATourneySeeds.csv")
w_reg     = w_reg[w_reg["Season"].isin(TRAIN_SEASONS)].copy()
w_tourney = w_tourney[w_tourney["Season"].isin(TRAIN_SEASONS)].copy()
print(f"  W Regular: {len(w_reg):,}  |  W Tourney: {len(w_tourney):,}  |  Massey: N/A")

# ─────────────────────────────────────────────────────────────
# RUN BOTH PIPELINES
# ─────────────────────────────────────────────────────────────
mens_preds = train_and_predict(
    m_reg, m_tourney, m_seeds, m_massey,
    MEN_LGBM, MEN_XGB,
    men_pairs,
    conf_csv       = "MTeamConferences.csv",
    seed_overrides = SEED_OVERRIDES_MENS,
    label          = "🏀 MEN'S",
)

womens_preds = train_and_predict(
    w_reg, w_tourney, w_seeds, None,
    WOMEN_LGBM, WOMEN_XGB,
    women_pairs,
    conf_csv       = "WTeamConferences.csv",
    seed_overrides = SEED_OVERRIDES_WOMENS,
    label          = "🏀 WOMEN'S",
)

# ─────────────────────────────────────────────────────────────
# STEP 8 — COMBINE, SAVE, SANITY CHECK
# ─────────────────────────────────────────────────────────────
print("\n" + "="*62)
print("  STEP 8 | Combining and saving submission …")
print("="*62)

# FIX ④ — map by ID dict (zero reindex drift)
all_preds      = {**mens_preds, **womens_preds}
stage2["Pred"] = stage2["ID"].map(all_preds)

n_missing = stage2["Pred"].isna().sum()
if n_missing:
    print(f"  ⚠️  {n_missing} IDs unmatched — filling with 0.5")
    stage2["Pred"] = stage2["Pred"].fillna(0.5)

stage2["Pred"] = stage2["Pred"].clip(0.025, 0.975).round(6)
submission = stage2[["ID","Pred"]]
submission.to_csv(OUTPUT, index=False)

# Sanity check
print(f"\n  {'─'*52}")
print(f"  File         : {OUTPUT}")
print(f"  Total rows   : {len(submission):,}  (expected 132,133) "
      f"{'✅' if len(submission)==132133 else '❌'}")
print(f"  Men's rows   : {men_mask.sum():,}")
print(f"  Women's rows : {women_mask.sum():,}")
print(f"  Any NaN      : {submission['Pred'].isna().sum()} "
      f"{'✅' if submission['Pred'].isna().sum()==0 else '❌'}")
print(f"  Pred min     : {submission['Pred'].min():.4f}  (floor=0.025)")
print(f"  Pred max     : {submission['Pred'].max():.4f}  (cap=0.975)")
print(f"  Pred mean    : {submission['Pred'].mean():.4f}  (should be ~0.5)")
print(f"  Pred std     : {submission['Pred'].std():.4f}  (>0.15 = good)")
print(f"  Unique vals  : {submission['Pred'].nunique():,}")
print(f"  {'─'*52}")
print(f"\n  First 5 rows:")
print(submission.head(5).to_string(index=False))
print(f"\n  Last 5 rows (women's):")
print(submission.tail(5).to_string(index=False))
print(f"\n✅  shaurya_final_1.csv ready — upload to Kaggle!")
print(f"\n  FIXES APPLIED:")
print(f"  ① 2026 seeds used for prediction features")
print(f"  ② Conference strength penalty (MEAC/SWAC = -150 ELO)")
print(f"  ③ Seed overrides: 1 vs 16 → min 97.5% for 1-seed")
print(f"  ④ ID dict mapping — no reindex drift")

🖥️  Using 8 CPU cores

  STEP 1 | Loading Stage 2 sample submission …
  Total rows   : 132,133
  Men's rows   : 66,430  (TeamID 1xxx)
  Women's rows : 65,703  (TeamID 3xxx)

  Loading Men's raw data …
  M Regular: 84,808  |  M Tourney: 1,001  |  Massey: yes
  Loading Women's raw data …
  W Regular: 81,708  |  W Tourney: 961  |  Massey: N/A

██████████████████████████████████████████████████████████████
  🏀 MEN'S PIPELINE
██████████████████████████████████████████████████████████████

  STEP 2 | Computing ELO …
          5,690 (season, team) ELO entries
  FIX ② | Applying conference strength penalty …
          3,766 team-seasons penalised
  STEP 3 | Building team stats, recent form, seeds …
          Team-season rows : 5,639
          Massey           : available
  STEP 4 | Building matchup feature matrix …
          Train rows : 85,809  |  Features : 172
  STEP 5 | Training LightGBM …
          ✓ LightGBM done
          Training XGBoost …
          ✓ XGBoost done
          Blend → LGB

In [7]:
teams = pd.read_csv("MTeams.csv")

teams[teams["TeamName"].str.contains("Michigan")]
teams[teams["TeamName"].str.contains("Howard")]

FileNotFoundError: [Errno 2] No such file or directory: 'MTeams.csv'